In [7]:
import jax
import jax.numpy as jnp
import numpy as np

from IPython.display import HTML
import matplotlib.animation as anim
import matplotlib.pyplot as plt
import plotly.graph_objects as go

import hj_reachability as hj
from hj_reachability.zg_solver import step_until_converged, ZGSolverSettings, step_until_converged_with_time, step_until_converged_save_every
from hj_reachability.zg_time_integration import third_order_tvd_rk_div_freeze
from hj_reachability.qp_controller import solve_two_stage_qp

from AdmissibleControlSet import AdmissibleControlSet, compute_ab_state, compute_ab_grid
from feasible_interval import feasible_interval_grid_1d

In [8]:
gamma = 0.3
umax = 1;
dynamics = hj.systems.DoubleInt(control_mode="min",
                                disturbance_mode="max",
                                u_max=umax,
                                gamma = gamma)
grid = hj.Grid.from_lattice_parameters_and_boundary_conditions(hj.sets.Box(np.array([-3., -3.]),
                                                                           np.array([3., 3.])),
                                                               (101, 101 ))
values = jnp.linalg.norm(grid.states, axis=-1) 
# print(grid.states.shape)

solver_settings = ZGSolverSettings(convergence_threshold=1e-2,
                                   divergence_threshold=10,
                                   value_postprocessor = hj.solver.static_obstacle(values),
                                    )

In [9]:
time = 0.
target_time = -10
# target_values =step_until_converged(solver_settings, dynamics, grid, time, values, target_time,
#                                    convergence_threshold=solver_settings.convergence_threshold,
#                                    progress_bar=True,)

save_times, save_values = step_until_converged_save_every(
    solver_settings,
    dynamics,
    grid,
    time,
    values,
    target_time,
    t_step=0.1,
    convergence_threshold=solver_settings.convergence_threshold,
    progress_bar=True,
)

"""
np.savez_compressed(
    "hj_result.npz",
    values=np.asarray(jax.device_get(target_values)),
    grid_min=np.asarray(grid.domain.lo),
    grid_max=np.asarray(grid.domain.hi),
    grid_shape=np.asarray(grid.shape),
    time=np.asarray(t),
)
"""

100%|#########################################################################| 10.0000/10.0 [00:02<00:00,  3.82sim_s/s]


'\nnp.savez_compressed(\n    "hj_result.npz",\n    values=np.asarray(jax.device_get(target_values)),\n    grid_min=np.asarray(grid.domain.lo),\n    grid_max=np.asarray(grid.domain.hi),\n    grid_shape=np.asarray(grid.shape),\n    time=np.asarray(t),\n)\n'

In [10]:
A_grid, b_grid, info = compute_ab_grid(
    value_function=save_values[10],
    grid=grid,
    dynamics=dynamics,
    gamma=gamma,
    u_range=umax,
)

u_low_grid, u_high_grid, feasible_mask = feasible_interval_grid_1d(
    A_grid, b_grid, u_range=umax
)

print("A_grid shape:", A_grid.shape)
print("b_grid shape:", b_grid.shape)
print("u_low_grid shape:", u_low_grid.shape)
print("u_high_grid shape:", u_high_grid.shape)
print("feasible_mask shape:", feasible_mask.shape)
print("number of feasible grid points:", feasible_mask.sum())


A_grid shape: (101, 101, 1)
b_grid shape: (101, 101)
u_low_grid shape: (101, 101)
u_high_grid shape: (101, 101)
feasible_mask shape: (101, 101)
number of feasible grid points: 4144


In [22]:
X = np.asarray(grid.states).reshape(-1, grid.states.shape[-1])   # or your own batch of states
t_list = np.asarray(save_times)
X_hist = [X.copy()]

for k in range(len(save_times)):
    t_now = save_times[k]
    A, b, info = compute_ab_state(
        value_function=save_values[len(save_times)-k],
        grid=grid,
        dynamics=dynamics,
        x=X,
        gamma=gamma,
        u_range=umax,
        times=save_times,
        t=t_now,
    ) 
    u_low, u_high, feasible_mask = feasible_interval_grid_1d(
    A, b, u_range=umax
    )

    u_mid = 0.5 * (u_low + u_high)
    u_mid = np.where(feasible_mask, u_mid, 0.0)

    def one_step(x, u):
            f = dynamics.open_loop_dynamics(x, t_now)
            G = dynamics.control_jacobian(x, t_now)   # shape (n,1)
            return f + G[:, 0] * u

    X_new = np.asarray(jax.vmap(one_step)(jnp.asarray(X), jnp.asarray(u_mid)))

    X = X_new
    X_hist.append(X.copy())
    


In [25]:
print(len(X_hist))
print(np.array(X_hist).shape)
V_X = jax.vmap(lambda x: grid.interpolate(save_values[k], x))(jnp.asarray(X))
print(np.array(V_X).shape)

102
(102, 10201, 2)
(10201,)
